<a href="https://colab.research.google.com/github/ploypailinnol106-netizen/reviews-text-notclean/blob/main/reviews_text_notclean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#test 1  model="distilbert-base-uncased-finetuned-sst-2-english"


In [ ]:
import pandas as pd
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

# ==== Config ====
FILE_PATH = '/mnt/reviews_text_notclean.xlsx'  # อัปเดตพาธไฟล์
TEXT_COLUMN = 'text'  # ปรับตามคอลัมน์จริง
SIM_THRESHOLD = 0.4

aspect_prompts = {
    "Itinerary": "Comments about the tour itinerary or plan",
    "Value for Money": "Comments about the price or value for money",
    "Tuk Tuk Quality": "Comments about the tuk tuk ride or comfort",
    "Tour Guide": "Comments about the tour guide or staff",
    "Food": "Comments about the food or meals"
}

# ==== Load Models ====
embedder = SentenceTransformer('all-MiniLM-L6-v2')
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# ==== Load Data ====
df = pd.read_excel(FILE_PATH)  # อ่านไฟล์ Excel

# ==== Process All Reviews ====
results = []

for idx, review in tqdm(df[TEXT_COLUMN].dropna().items(), total=len(df), desc="Processing Reviews"):
    # 1. Split review into sub-sentences
    sub_sentences = [s.strip() for s in re.split(r'[.;]| and | but ', review) if s.strip()]
    if not sub_sentences:
        continue

    sub_embeddings = embedder.encode(sub_sentences, convert_to_tensor=True)
    aspect_embeddings = embedder.encode(list(aspect_prompts.values()), convert_to_tensor=True)

    aspect_result = {"review_index": idx, "original_review": review}

    # 2. For each aspect
    for i, aspect in enumerate(aspect_prompts):
        aspect_vec = aspect_embeddings[i]
        cosine_scores = util.cos_sim(aspect_vec, sub_embeddings)[0]
        max_score = cosine_scores.max().item()
        best_idx = int(cosine_scores.argmax())

        if max_score > SIM_THRESHOLD:
            sentence = sub_sentences[best_idx]
            sentiment = sentiment_analyzer(sentence)[0]
            aspect_result[f"{aspect}_sentence"] = sentence
            aspect_result[f"{aspect}_sentiment"] = sentiment['label']
            aspect_result[f"{aspect}_score"] = sentiment['score']
        else:
            aspect_result[f"{aspect}_sentence"] = None
            aspect_result[f"{aspect}_sentiment"] = None
            aspect_result[f"{aspect}_score"] = None

    results.append(aspect_result)

# ==== Create DataFrame with Results ====
result_df = pd.DataFrame(results)

# ==== Save or Display ====
output_path = "/mnt/aspect_sentiment_output.xlsx"
result_df.to_excel(output_path, index=False)

print(f"✅ บันทึกไฟล์ผลลัพธ์ที่: {output_path}")
print(result_df.head())


Device set to use cpu
Processing Reviews: 100%|██████████| 14682/14682 [1:20:20<00:00,  3.05it/s]


✅ บันทึกไฟล์ผลลัพธ์ที่: /mnt/aspect_sentiment_output.xlsx
   review_index                                    original_review  \
0             0  The tut tuts were so much fun to ride around i...   
1             1  A truly fantastic, fun experience that we woul...   
2             2  We had so much fun and the food was absolutely...   
3             3  A really good tour with excellent food and a k...   
4             4  Amazing tour! I really enjoyed the experience ...   

                                  Itinerary_sentence Itinerary_sentiment  \
0  you got to see a lot of the city during this tour            POSITIVE   
1                                               None                None   
2                 I highly recommend doing this tour            POSITIVE   
3             A really good tour with excellent food            POSITIVE   
4  Amazing tour! I really enjoyed the experience ...            POSITIVE   

   Itinerary_score   Value for Money_sentence Value for Money_se

#test 2 model="cardiffnlp/twitter-roberta-base-sentiment"

In [ ]:
# ==== Install dependencies (for Colab) ====
!pip install -q pandas openpyxl nltk sentence-transformers transformers tqdm

# ==== Import libraries ====
import pandas as pd
import re
import nltk
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from nltk.tokenize import sent_tokenize

# ==== Download NLTK Data ====
nltk.download('punkt')
# Download the 'punkt_tab' data package
nltk.download('punkt_tab') # Add this line to download the required data

# ==== Config ====
FILE_PATH = '/content/reviews_text_notclean.xlsx'  # เปลี่ยนตาม path จริงถ้าใช้บน Colab
TEXT_COLUMN = 'text'  # ชื่อคอลัมน์ที่เก็บรีวิว
SIM_THRESHOLD = 0.5   # ปรับให้แม่นยำขึ้น

# ==== Aspect Prompts (เขียนใหม่ให้ชัดเจนขึ้น) ====
aspect_prompts = {
    "Itinerary": "The review talks about the schedule, plan, or activities of the tour.",
    "Value for Money": "The review talks about the price, cost, or whether the experience was worth the money.",
    "Tuk Tuk Quality": "The review talks about the tuk tuk vehicle, ride quality, or comfort.",
    "Tour Guide": "The review talks about the staff, guide, friendliness, or helpfulness.",
    "Food": "The review talks about the food, meals, or dining experience."
}

# ==== Load Models ====
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# ✅ ใช้ sentiment model ที่เหมาะกับรีวิวมากขึ้น (Twitter RoBERTa)
sentiment_analyzer = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

# ==== Load Data ====
df = pd.read_excel(FILE_PATH)

# ==== Process All Reviews ====
results = []

for idx, review in tqdm(df[TEXT_COLUMN].dropna().items(), total=len(df), desc="Processing Reviews"):
    # 1. Split review into sub-sentences using nltk
    sub_sentences = sent_tokenize(review)
    if not sub_sentences:
        continue

    sub_embeddings = embedder.encode(sub_sentences, convert_to_tensor=True)
    aspect_embeddings = embedder.encode(list(aspect_prompts.values()), convert_to_tensor=True)

    aspect_result = {"review_index": idx, "original_review": review}

    # 2. For each aspect
    for i, aspect in enumerate(aspect_prompts):
        aspect_vec = aspect_embeddings[i]
        cosine_scores = util.cos_sim(aspect_vec, sub_embeddings)[0]
        max_score = cosine_scores.max().item()
        best_idx = int(cosine_scores.argmax())

        if max_score > SIM_THRESHOLD:
            sentence = sub_sentences[best_idx]
            sentiment = sentiment_analyzer(sentence)[0]
            aspect_result[f"{aspect}_sentence"] = sentence
            aspect_result[f"{aspect}_sentiment"] = sentiment['label']
            aspect_result[f"{aspect}_score"] = sentiment['score']
        else:
            aspect_result[f"{aspect}_sentence"] = None
            aspect_result[f"{aspect}_sentiment"] = None
            aspect_result[f"{aspect}_score"] = None

    results.append(aspect_result)

# ==== Save Results ====
result_df = pd.DataFrame(results)
output_path = "/content/aspect_sentiment_output_updated.xlsx"
result_df.to_excel(output_path, index=False)

print(f"✅ Results saved to: {output_path}")
result_df.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
Device set to use cpu
Processing Reviews: 100%|██████████| 14682/14682 [1:05:12<00:00,  3.75it/s]


✅ Results saved to: /content/aspect_sentiment_output_updated.xlsx


,review_index,original_review,Itinerary_sentence,Itinerary_sentiment,Itinerary_score,Value for Money_sentence,Value for Money_sentiment,Value for Money_score,Tuk Tuk Quality_sentence,Tuk Tuk Quality_sentiment,Tuk Tuk Quality_score,Tour Guide_sentence,Tour Guide_sentiment,Tour Guide_score,Food_sentence,Food_sentiment,Food_score
0,0,The tut tuts were so much fun to ride around i...,None,None,NaN,None,None,NaN,None,None,NaN,None,None,NaN,None,None,NaN
1,1,"A truly fantastic, fun experience that we woul...",None,None,NaN,"A very good price, for an even better amazing ...",LABEL_2,0.980187,None,None,NaN,None,None,NaN,Passing through (and eating and fasting) Miche...,LABEL_2,0.526898
2,2,We had so much fun and the food was absolutely...,I highly recommend doing this tour.,LABEL_2,0.968167,None,None,NaN,None,None,NaN,None,None,NaN,None,None,NaN
3,3,A really good tour with excellent food and a k...,A really good tour with excellent food and a k...,LABEL_2,0.978942,None,None,NaN,None,None,NaN,None,None,NaN,A really good tour with excellent food and a k...,LABEL_2,0.978942
4,4,Amazing tour! I really enjoyed the experience ...,Amazing tour!,LABEL_2,0.969437,None,None,NaN,None,None,NaN,None,None,NaN,None,None,NaN


#test3 for i, aspect in enumerate(aspect_prompts):

In [ ]:
# ==== Install dependencies (for Colab) ====
!pip install -q pandas openpyxl nltk sentence-transformers transformers tqdm

# ==== Import libraries ====
import pandas as pd
import nltk
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

# ==== Download NLTK Data ====
nltk.download('punkt')

# ==== Config ====
FILE_PATH = '/content/reviews_text_notclean.xlsx'  # เปลี่ยนตาม path จริง
TEXT_COLUMN = 'text'  # ชื่อคอลัมน์ที่เก็บรีวิว
SIM_THRESHOLD = 0.4   # ความคล้ายขั้นต่ำ

# ==== Aspect Prompts (เขียนใหม่ให้ละเอียดขึ้น) ====
aspect_prompts = {
    "Itinerary": "The review talks about the schedule, plan, or activities of the tour.",
    "Value for Money": "The review talks about the price, cost, or whether the experience was worth the money.",
    "Tuk Tuk Quality": "The review talks about the tuk tuk vehicle, ride quality, or comfort.",
    "Tour Guide": "The review talks about the staff, guide, friendliness, or helpfulness.",
    "Food": "The review talks about the food, meals, or dining experience."
}

# ==== Load Models ====
embedder = SentenceTransformer('all-MiniLM-L6-v2')
sentiment_analyzer = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

# แปลง Label ให้ human-readable
label_map = {
    'LABEL_0': 'NEGATIVE',
    'LABEL_1': 'NEUTRAL',
    'LABEL_2': 'POSITIVE'
}

# ==== Load Data ====
df = pd.read_excel(FILE_PATH)

# ==== Process All Reviews ====
results = []

for idx, review in tqdm(df[TEXT_COLUMN].dropna().items(), total=len(df), desc="🔍 Processing Reviews"):
    # 1. แยกประโยค
    sub_sentences = sent_tokenize(review)
    if not sub_sentences:
        continue

    sub_embeddings = embedder.encode(sub_sentences, convert_to_tensor=True)
    aspect_embeddings = embedder.encode(list(aspect_prompts.values()), convert_to_tensor=True)

    aspect_result = {"review_index": idx, "original_review": review}

    # 2. ตรวจแต่ละ aspect
    for i, aspect in enumerate(aspect_prompts):
        aspect_vec = aspect_embeddings[i]
        cosine_scores = util.cos_sim(aspect_vec, sub_embeddings)[0]

        matching_sentences = []
        sentiments = []

        for j, score in enumerate(cosine_scores):
            if score.item() > SIM_THRESHOLD:
                sent_text = sub_sentences[j]
                sent_result = sentiment_analyzer(sent_text)[0]
                sentiment_label = label_map.get(sent_result['label'], sent_result['label'])
                matching_sentences.append(sent_text)
                sentiments.append((sentiment_label, sent_result['score']))

        if matching_sentences:
            aspect_result[f"{aspect}_sentence"] = " | ".join(matching_sentences)
            aspect_result[f"{aspect}_sentiment"] = ", ".join([s[0] for s in sentiments])
            aspect_result[f"{aspect}_score"] = ", ".join([f"{s[1]:.2f}" for s in sentiments])
        else:
            aspect_result[f"{aspect}_sentence"] = None
            aspect_result[f"{aspect}_sentiment"] = None
            aspect_result[f"{aspect}_score"] = None

    results.append(aspect_result)

# ==== Save Results ====
result_df = pd.DataFrame(results)
output_path = "/content/aspect_sentiment_output_test3.xlsx"
result_df.to_excel(output_path, index=False)

print(f"✅ Results saved to: {output_path}")
result_df.head()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Device set to use cpu
🔍 Processing Reviews: 100%|██████████| 14682/14682 [2:11:19<00:00,  1.86it/s]


✅ Results saved to: /content/aspect_sentiment_output_test3.xlsx


,review_index,original_review,Itinerary_sentence,Itinerary_sentiment,Itinerary_score,Value for Money_sentence,Value for Money_sentiment,Value for Money_score,Tuk Tuk Quality_sentence,Tuk Tuk Quality_sentiment,Tuk Tuk Quality_score,Tour Guide_sentence,Tour Guide_sentiment,Tour Guide_score,Food_sentence,Food_sentiment,Food_score
0,0,The tut tuts were so much fun to ride around i...,The guide was very personable and made the tri...,POSITIVE,0.98,None,None,None,The tut tuts were so much fun to ride around i...,POSITIVE,0.98,None,None,None,I personally didn’t love the food but I found ...,NEGATIVE,0.92
1,1,"A truly fantastic, fun experience that we woul...",None,None,None,"A very good price, for an even better amazing ...",POSITIVE,0.98,"We went to authentically Thai food places, win...",POSITIVE,0.97,None,None,None,Passing through (and eating and fasting) Miche...,POSITIVE,0.53
2,2,We had so much fun and the food was absolutely...,"Our tour was basically a private tour, very sm...","NEUTRAL, POSITIVE","0.78, 0.97",None,None,None,None,None,None,None,None,None,We had so much fun and the food was absolutely...,"POSITIVE, POSITIVE, POSITIVE, POSITIVE","0.99, 0.97, 0.88, 0.84"
3,3,A really good tour with excellent food and a k...,A really good tour with excellent food and a k...,POSITIVE,0.98,Very good value for money.,POSITIVE,0.92,None,None,None,None,None,None,A really good tour with excellent food and a k...,POSITIVE,0.98
4,4,Amazing tour! I really enjoyed the experience ...,Amazing tour!,POSITIVE,0.97,None,None,None,None,None,None,None,None,None,None,None,None


#test4

In [ ]:
# ==== Install dependencies (for Colab) ====
!pip install -q pandas openpyxl nltk sentence-transformers transformers tqdm

# ==== Import Libraries ====
import pandas as pd
import nltk
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

# ==== Download NLTK Data ====
nltk.download('punkt')
nltk.download('punkt_tab') # Download the punkt_tab data package

# ==== Config ====
FILE_PATH = '/content/Data.xlsx'  # เปลี่ยนเป็น path จริงของไฟล์
TEXT_COLUMN = 'text'  # ชื่อคอลัมน์ที่มีรีวิว
SIM_THRESHOLD = 0.3   # ค่าความคล้ายขั้นต่ำในการจับ aspect

# ==== Aspect Prompts ====
aspect_prompts = {
    "Itinerary": "The review talks about the schedule, plan, or activities of the tour, including sightseeing and timing.",
    "Value for Money": "The review talks about the price, cost, or whether the experience was worth the money paid.",
    "Tuk Tuk Quality": "The review talks about the tuk tuk vehicle, ride quality, driving, tuk tuk driver, or comfort during the ride.",
    "Tour Guide": "The review talks about the tour guide, guide, staff, or specific guide such as Adam, Alice, Amp, Bill, Cat, Fern, Gi, Gimao, Ice, Jong, Tum, Joomboom, Khim, May, Mod, Nina, Nuch, Nudi, Neung, Preme, Tukta, Frien, Fon, Nat or others. It mentions friendliness, helpfulness, enthusiasm, or personality.",
    "Food": "The review talks about the food, meals, dishes, snacks, or overall dining experience during the tour."
}

# ==== Load Models ====
embedder = SentenceTransformer('all-MiniLM-L6-v2')
sentiment_analyzer = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

# ==== Sentiment Label Map ====
label_map = {
    'LABEL_0': 'NEGATIVE',
    'LABEL_1': 'NEUTRAL',
    'LABEL_2': 'POSITIVE'
}

# ==== Load Data ====
df = pd.read_excel(FILE_PATH)

# ==== Process Reviews ====
results = []

for idx, review in tqdm(df[TEXT_COLUMN].dropna().items(), total=len(df), desc="🔍 Processing Reviews"):
    try:
        sub_sentences = sent_tokenize(review)
    except Exception as e:
        print(f"❌ Error in sentence tokenization at index {idx}: {e}")
        continue

    if not sub_sentences:
        continue

    sub_embeddings = embedder.encode(sub_sentences, convert_to_tensor=True)
    aspect_embeddings = embedder.encode(list(aspect_prompts.values()), convert_to_tensor=True)

    aspect_result = {"review_index": idx, "original_review": review}

    for i, (aspect, prompt_text) in enumerate(aspect_prompts.items()):
        aspect_vec = aspect_embeddings[i]
        cosine_scores = util.cos_sim(aspect_vec, sub_embeddings)[0]

        max_confidence = 0.0
        best_sentiment = None
        best_sentence_list = []

        for j, score in enumerate(cosine_scores):
            if score.item() > SIM_THRESHOLD:
                sentence = sub_sentences[j]
                sent_result = sentiment_analyzer(sentence)[0]
                sentiment_label = label_map.get(sent_result['label'], sent_result['label'])
                confidence = sent_result['score']

                if sentiment_label != "NEUTRAL" and confidence > max_confidence:
                    max_confidence = confidence
                    best_sentiment = sentiment_label
                    best_sentence_list = [sentence]
                elif sentiment_label != "NEUTRAL" and confidence == max_confidence:
                    best_sentence_list.append(sentence)  # กรณีคะแนนเท่ากัน

        # Save best sentiment
        if best_sentiment:
            aspect_result[f"{aspect}_sentiment"] = best_sentiment
            aspect_result[f"{aspect}_score"] = f"{max_confidence:.2f}"
            aspect_result[f"{aspect}_sentence"] = " ".join(best_sentence_list)
        else:
            aspect_result[f"{aspect}_sentiment"] = None
            aspect_result[f"{aspect}_score"] = None
            aspect_result[f"{aspect}_sentence"] = None

    results.append(aspect_result)

# ==== Save Output ====
result_df = pd.DataFrame(results)
output_path = "/content/Data_output.xlsx"
result_df.to_excel(output_path, index=False)

print(f"\n✅ Results saved to: {output_path}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Device set to use cpu
🔍 Processing Reviews: 100%|██████████| 3578/3578 [1:31:33<00:00,  1.54s/it]



✅ Results saved to: /content/Data_output.xlsx
